In [1]:
### Building a RAG System with LangChain and ChromaDB
#### Introduction
"""Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)"""

'Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:\n\n- LangChain: A framework for developing applications powered by language models\n- ChromaDB: An open-source vector database for storing and retrieving embeddings\n- OpenAI: For embeddings and language model (you can substitute with other providers)'

In [94]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
### langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document


## vectorstores
from langchain_community.vectorstores import Chroma

## utility imports
import numpy as np
from typing import List


d:\Projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



In [4]:
### 1. Sample Data

In [5]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [6]:
## save sample documents to files
import tempfile
temp_dir = tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)

print(f"Sample document create in {temp_dir}")

Sample document create in C:\Users\TANMAY~1\AppData\Local\Temp\tmpirw245p6


In [7]:
## save sample documents to files
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt","w") as f:
        f.write(doc)

In [8]:
### 2. Document Loading

In [9]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader

# Load documents from directory
loader = DirectoryLoader(
    "data", 
    glob="*.txt", 
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)
documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:200] + "...")

Loaded 3 documents

First document preview:

    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. Ther...


In [10]:
### Document Splitting

In [11]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Maximum size of each chunk
    chunk_overlap=50,  # Overlap between chunks to maintain context
    length_function=len,
    separators=[" "]  # Hierarchy of separators
)
chunks=text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 5 chunks from 3 documents

Chunk example:
Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experie...
Metadata: {'source': 'data\\doc_0.txt'}


In [12]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of in

In [13]:
### Embedding Models
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [14]:
sample_text = "Machine Learning is fascinating"
embeddings= OpenAIEmbeddings()
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x00000186A2A93E60>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x00000186A2A90680>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [15]:
vector = embeddings.embed_query(sample_text)
vector

[-0.02172444760799408,
 0.016208980232477188,
 0.010213345289230347,
 -0.022516079246997833,
 -0.0037213172763586044,
 0.01783117651939392,
 4.82096329506021e-05,
 0.01027174387127161,
 -0.015547124668955803,
 -0.04134652763605118,
 0.007929293438792229,
 0.03628527745604515,
 -0.019128933548927307,
 -0.008234266191720963,
 -0.0013058676850050688,
 0.00581719446927309,
 0.03880292549729347,
 0.008811768144369125,
 -0.0005584409227594733,
 -0.008591149002313614,
 -0.031224025413393974,
 0.022048886865377426,
 -0.005914526060223579,
 -0.03441650792956352,
 -0.014898247085511684,
 0.0023018959909677505,
 0.003834871109575033,
 -0.03885483369231224,
 -0.012523352168500423,
 -0.002739888848736882,
 0.027590306475758553,
 -0.004736811853945255,
 -0.0170655008405447,
 -0.03981517627835274,
 -0.008513283915817738,
 -0.012211889959871769,
 -0.004152821376919746,
 0.0028583090752363205,
 -0.01670212857425213,
 -0.00013068816042505205,
 0.020076295360922813,
 0.02541007660329342,
 -0.008435418829

In [16]:
### Initialize the ChromaDB Vector Store and Stores the chinks in vector Representation

In [17]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of in

In [18]:
## Create a Chromadb Vector Store 
persist_directory  ="./chroma_db"

## Initialize CHromaDB with Open AI embeddings
vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = OpenAIEmbeddings(),
    persist_directory = persist_directory,
    collection_name = "rag_collection"
)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to {persist_directory}")

Vector store created with 15 vectors
Persisted to ./chroma_db


In [19]:
### Test Similarity Search

In [20]:
query = "What are the types of machine learning ?"

similar_docs = vectorstore.similarity_search(query, k = 3)
similar_docs

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, w

In [21]:
query = "What is NLP?"

similar_docs = vectorstore.similarity_search(query, k = 3)
similar_docs

[Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mec

In [22]:
query = "What is Deep Learning?"

similar_docs = vectorstore.similarity_search(query, k = 3)
similar_docs

[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neura

In [23]:
print(f"Query : {query}")
print(f"\nTop {len(similar_docs)} similar_chunks:")
for i,doc in enumerate(similar_docs):
    print(f"\n --- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source : {doc.metadata.get('source', 'Unknown')}")

Query : What is Deep Learning?

Top 3 similar_chunks:

 --- Chunk 1 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...
Source : data\doc_1.txt

 --- Chunk 2 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...
Source : data\doc_1.txt

 --- Chunk 3 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...
Source : data\doc_1.txt


In [24]:
### Advanced Similarity Search with Scores

In [25]:
results_scores = vectorstore.similarity_search_with_score(query, k = 3)
results_scores

[(Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
  0.22442439198493958),
 (Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recogn

In [26]:
### Understanding Similarity Scroes
""" The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)"""

' The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:\n\nChromaDB default: Uses L2 distance (Euclidean distance)\n\n- Lower scores = MORE similar (closer in vector space)\n- Score of 0 = identical vectors\n- Typical range: 0 to 2 (but can be higher)\n\n\nCosine similarity (if configured):\n\n- Higher scores = MORE similar\n- Range: -1 to 1 (1 being identical)'

In [27]:
### Initialize LLM, RAG Chain, Prompt Template,  Query the RAG system

In [28]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name =  "gpt-3.5-turbo",
    temperature  = 0.2,
    max_tokens = 500
)

In [29]:
test_response = llm.invoke("What is Large Language Models ")

test_response

AIMessage(content="Large Language Models (LLMs) are a type of artificial intelligence model that is trained on vast amounts of text data to understand and generate human-like language. These models are capable of performing a wide range of natural language processing tasks, such as text generation, translation, summarization, and more. LLMs have become increasingly popular in recent years due to their ability to generate coherent and contextually relevant text, making them valuable tools for a variety of applications in fields such as natural language understanding, chatbots, and content generation. Examples of popular LLMs include OpenAI's GPT-3 and Google's BERT.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 126, 'prompt_tokens': 13, 'total_tokens': 139, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 

In [30]:
from langchain.chat_models.base import init_chat_model

llm  = init_chat_model("openai:gpt-3.5-turbo")
#llm = init_chat_model("groq:")
llm

ChatOpenAI(profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': True, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000186A70A8F80>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000186A70A8530>, root_client=<openai.OpenAI object at 0x00000186A45E93A0>, root_async_client=<openai.AsyncOpenAI object at 0x00000186A70A8920>, model_kwargs={}, openai_api_key=SecretStr('**********'),

In [31]:
llm.invoke("What is AI")

AIMessage(content='AI, or artificial intelligence, refers to the simulation of human intelligence in machines or computer systems. These intelligent machines are designed to perform tasks that typically require human intelligence, such as learning, reasoning, problem-solving, perception, and language understanding. AI technologies include machine learning, deep learning, natural language processing, and neural networks. AI is increasingly being used in various industries, including healthcare, finance, transportation, and entertainment, to automate processes, improve efficiency, and provide new insights.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 10, 'total_tokens': 106, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name':

In [34]:
### Modern RAG Chain
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [35]:
from langchain_core.prompts import ChatPromptTemplate


In [36]:
## Convert vector store to retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3},  # top 3 chunks (was misspelled search_kwarg)
)
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000186A1821A30>, search_kwargs={})

In [ ]:
# LangChain 1.x: retrievers are Runnables — use .invoke(query), not .get_relevant_documents(query)
retriever.invoke("What is Deep Learning")

In [37]:
## Create a prompt template
from langchain_core.prompts import ChatPromptTemplate
system_prompt="""You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [38]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [39]:
# Create a Document chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain = create_stuff_documents_chain
document_chain

<function langchain_classic.chains.combine_documents.stuff.create_stuff_documents_chain(llm: langchain_core.runnables.base.Runnable[langchain_core.prompt_values.PromptValue | str | collections.abc.Sequence[langchain_core.messages.base.BaseMessage | list[str] | tuple[str, str] | str | dict[str, typing.Any]], langchain_core.messages.base.BaseMessage | str], prompt: langchain_core.prompts.base.BasePromptTemplate, *, output_parser: langchain_core.output_parsers.base.BaseOutputParser | None = None, document_prompt: langchain_core.prompts.base.BasePromptTemplate | None = None, document_separator: str = '\n\n', document_variable_name: str = 'context') -> langchain_core.runnables.base.Runnable[dict[str, typing.Any], typing.Any]>

In [40]:
# Create a Document chain using the modern runnable API
from langchain_core.output_parsers import StrOutputParser

# Create a simple document chain that combines prompt + LLM
document_chain = prompt | llm | StrOutputParser()
document_chain

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': Fals

In [41]:
## Create a RAG Pipeline using LCEL (Langchain Expression Language)

In [42]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [43]:
# Create a custom prompt.

custom_prompt = ChatPromptTemplate.from_template(""" Use the following context to answer the question.
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.


Context: {context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template=" Use the following context to answer the question.\nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [44]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000186A1821A30>, search_kwargs={})

In [45]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [49]:
## Build the chain using LCEL
rag_chain_lcel  = (
    {
        "context":retriever | format_docs, 
        "question" : RunnablePassthrough()}
    | custom_prompt
    | llm
    | StrOutputParser()
)


rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000186A1821A30>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template=" Use the following context to answer the question.\nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| ChatOpenAI(profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs

In [54]:
response = rag_chain_lcel.invoke("What is Deep Learning")
response

'Deep learning is a subset of machine learning based on artificial neural networks. These networks consist of layers of interconnected nodes and are inspired by the human brain. Deep learning has revolutionized fields such as computer vision, natural language processing, and speech recognition. Additionally, convolutional neural networks (CNNs) are effective for image processing, while recurrent neural networks (RNNs) and transformers are also commonly used in deep learning.'

In [57]:
retriever.invoke("What is Deep Learning")

[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neura

In [60]:
# Query using the LCEL approach -Fixed Version
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)

    # Method 1 : Pass string directly (when using Runnable Passthroug)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")

    # Get source documents separately if needed
    docs = retriever.invoke(question)
    print("\n Source Documents")
    for i, doc in enumerate(docs):
        print(f"\n --- Source {i + 1} ---")
        print(doc.page_content[:200] + "...")

In [61]:
# Test LCEL chain
print("Testing LCEL Chain")
query_rag_lcel("What are the key concepts in reinforcement learning?")

Testing LCEL Chain
Question: What are the key concepts in reinforcement learning?
--------------------------------------------------
Answer: The key concepts in reinforcement learning are learning through interaction with an environment, using rewards and penalties.

 Source Documents

 --- Source 1 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

 --- Source 2 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

 --- Source 3 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

 --- Source 4 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...


In [62]:
## Add New Documents to existing Vector Store

In [63]:
vectorstore

In [64]:
# Add new docuemtns to the existing vector store
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties
based on its actions and learns to maximize cumulative reward over time. Key concepts
 in RL include: states, actions, rewards, policies and value functions.  Popular RL algorithms include:
- Q-learning
- Deep Q-Network (DQN)
- Policy Gradient methods
- Actor-Critic methods  RL has been successfully applied to game playing like AlphaGo, robotics, and autonomous systems."""

In [65]:
new_document

'\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties\nbased on its actions and learns to maximize cumulative reward over time. Key concepts\n in RL include: states, actions, rewards, policies and value functions.  Popular RL algorithms include:\n- Q-learning\n- Deep Q-Network (DQN)\n- Policy Gradient methods\n- Actor-Critic methods  RL has been successfully applied to game playing like AlphaGo, robotics, and autonomous systems.'

In [66]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of in

In [67]:
new_doc = Document(
    page_content = new_document,
    metadata = {"source":"manual_addition", "topic":"reinforcement_learning"}
)

In [68]:
new_doc

Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties\nbased on its actions and learns to maximize cumulative reward over time. Key concepts\n in RL include: states, actions, rewards, policies and value functions.  Popular RL algorithms include:\n- Q-learning\n- Deep Q-Network (DQN)\n- Policy Gradient methods\n- Actor-Critic methods  RL has been successfully applied to game playing like AlphaGo, robotics, and autonomous systems.')

In [70]:
## split the documents
new_chunks = text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='Reinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties\nbased on its actions and learns to maximize cumulative reward over time. Key concepts\n in RL include: states, actions, rewards, policies and value functions.  Popular RL algorithms include:\n- Q-learning\n- Deep Q-Network (DQN)\n- Policy Gradient methods\n- Actor-Critic methods  RL has been'),
 Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='methods\n- Actor-Critic methods  RL has been successfully applied to game playing like AlphaGo, robotics, and autonomous systems.')]

In [71]:
### Add new documents to vectorstore
vectorstore.add_documents(new_chunks)

['bb07538a-960a-433e-8194-9a2d69fcd76e',
 '3bf4d07d-b6cf-42c0-8391-e4c7e7175d80']

In [72]:
print(f"Added {len(new_chunks)} new chunks to the vector store")
print(f"Total vectors now: {vectorstore._collection.count()}")

Added 2 new chunks to the vector store
Total vectors now: 17


In [73]:
## query with the updated vector 
new_question = "What are the key concepts in reinforcement learnging"
result = query_rag_lcel(new_question)
new_question

Question: What are the key concepts in reinforcement learnging
--------------------------------------------------
Answer: The key concepts in reinforcement learning are states, actions, rewards, policies, and value functions.

 Source Documents

 --- Source 1 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or p...

 --- Source 2 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

 --- Source 3 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

 --- Source 4 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....


'What are the key concepts in reinforcement learnging'

In [74]:
### Advanced RAG techniques with conversational memory

In [77]:
"""- create_history_aware_retriever : Makes the retriever understand conversation context
- MessagesPlaceholder : Placeholder for chat history in prompts
- HumanMessage/AIMessage: Structured message types for conversation history"""

'- create_history_aware_retriever : Makes the retriever understand conversation context\n- MessagesPlaceholder : Placeholder for chat history in prompts\n- HumanMessage/AIMessage: Structured message types for conversation history'

In [79]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import  HumanMessage, AIMessage

In [82]:
## create a prompt that includes the chat history
contextualize_q_system_prompt =  """"Given a chat history and the latest user question 
which might reference context in the chat history, formulate a stand-alone question 
which can be understood without the chat history. Do not answer the question; 
just reformulate it if needed and otherwise return it as is."""


contextualize_q_prompt  = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

In [83]:
## create history aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000186A1821A30>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIM

In [84]:
# Create a new document chain with history
qa_system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

question_answer_chain = create_stuff_documents_chain(llm,qa_prompt)


# Create conversation RAG chain
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)

print("Conversational RAG chain created!")

Conversational RAG chain created!


In [85]:
chat_history = []
# First question
result1 = conversational_rag_chain.invoke({
    "chat_history":chat_history,
    "input": "What is machine learning?"
})

print(f"Q: What is machine learning?")
print(f"A: {result1['answer']}")

Q: What is machine learning?
A: Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data.


In [87]:
chat_history.extend([
    HumanMessage(content="What is Machine learning"),
    AIMessage(content = result1['answer'])
])

In [89]:
#Follow up question
result2 = conversational_rag_chain.invoke({
    "chat_history":chat_history,
    "input": "What are its main types?"  # Refers to ML from previous question
})
result2

{'chat_history': [HumanMessage(content='What is Machine learning', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'input': 'What are its main types?',
 'context': [Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervis

In [90]:
result2['answer']

'The main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, unsupervised learning finds patterns in unlabeled data, and reinforcement learning learns through a system of rewards and punishments.'

In [91]:
## Using GROQLLM's

In [92]:
llm

ChatOpenAI(profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': True, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000186A70A8F80>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000186A70A8530>, root_client=<openai.OpenAI object at 0x00000186A45E93A0>, root_async_client=<openai.AsyncOpenAI object at 0x00000186A70A8920>, model_kwargs={}, openai_api_key=SecretStr('**********'),

In [95]:
load_dotenv()

True

In [96]:
os.getenv("GROQ_API_KEY")

'***REDACTED***'

In [ ]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

In [ ]:
os.environ["GROQ_API_KEY"]

In [ ]:
llm = ChatGroq(model = "gemma2-9b-it")